# 🏠 SmartLease Edge — Multi-Class Wall Defect Detection

Train **YOLOv8n** (detection) on 5 wall defect classes:
- `crack` · `mold` · `peeling_paint` · `stairstep_crack` · `water_seepage`

### Steps:
1. GPU Check & Install
2. Download Roboflow Dataset (5 classes, bbox annotated)
3. Merge with existing crack dataset (optional)
4. Train YOLOv8n
5. Evaluate & Visualize
6. Export (ONNX) & Download

## 1. Environment & GPU Check

In [ ]:
!pip install -q ultralytics roboflow opencv-python matplotlib pyyaml

import torch, cv2, os, shutil, random, yaml
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## 2. Download Multi-Class Dataset from Roboflow

**Get your free API key:** https://app.roboflow.com/settings/api

Dataset: [building-defect](https://universe.roboflow.com/builddef2/building-defect-mmjsi) — 5 classes, bbox annotated

In [ ]:
# ===== PASTE YOUR FREE ROBOFLOW API KEY HERE =====
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # Get from https://app.roboflow.com/settings/api

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("builddef2").project("building-defect-mmjsi")
version = project.version(3)
dataset = version.download("yolov8", location="/content/roboflow-defects")

ROBOFLOW_DIR = Path(dataset.location)
print(f"\n✅ Downloaded to: {ROBOFLOW_DIR}")

# Show data.yaml
data_yaml = ROBOFLOW_DIR / "data.yaml"
with open(data_yaml) as f:
    config = yaml.safe_load(f)
    print(f"\nClasses ({config['nc']}): {config['names']}")

## 2b. (Optional) Merge with Your Existing Crack Dataset

Upload your `wall-defects` folder (the one with crack segmentation data) as a ZIP to Colab.

**Skip this cell if you only want to train on the Roboflow dataset.**

In [ ]:
# ===== OPTIONAL: Upload your existing crack dataset ZIP =====
# 1. Zip your datasets/wall-defects folder on your PC
# 2. Upload it to Colab using the file browser (left sidebar)
# 3. Set the path below

MERGE_CRACK_DATA = False  # Set to True if you uploaded your crack dataset
CRACK_ZIP_PATH = "/content/wall-defects.zip"  # Path to uploaded zip

if MERGE_CRACK_DATA and os.path.exists(CRACK_ZIP_PATH):
    import zipfile
    CRACK_DIR = Path("/content/wall-defects-crack")
    with zipfile.ZipFile(CRACK_ZIP_PATH, 'r') as zf:
        zf.extractall(str(CRACK_DIR))
    
    # Find the actual data directory
    for candidate in [CRACK_DIR, CRACK_DIR / "wall-defects"]:
        if (candidate / "train" / "images").exists():
            CRACK_DIR = candidate
            break
    
    print(f"Crack dataset extracted to: {CRACK_DIR}")
    
    # Convert seg polygon labels → bbox and merge into Roboflow train
    crack_class_id = config['names'].index('crack') if 'crack' in config['names'] else 0
    print(f"Crack maps to class_id: {crack_class_id}")
    
    merged_count = 0
    for split in ['train', 'valid', 'test']:
        src_img = CRACK_DIR / split / 'images'
        src_lbl = CRACK_DIR / split / 'labels'
        if not src_img.exists():
            continue
        
        # Always merge into train (we'll re-split later)
        dst_img = ROBOFLOW_DIR / 'train' / 'images'
        dst_lbl = ROBOFLOW_DIR / 'train' / 'labels'
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        
        for img_file in src_img.iterdir():
            if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            
            new_name = f"crack_{img_file.name}"
            shutil.copy2(img_file, dst_img / new_name)
            
            lbl_file = src_lbl / (img_file.stem + '.txt')
            dst_lbl_file = dst_lbl / f"crack_{img_file.stem}.txt"
            
            if lbl_file.exists():
                with open(lbl_file) as f:
                    lines = f.readlines()
                
                det_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    coords = [float(p) for p in parts[1:]]
                    if len(coords) == 4:
                        cx, cy, w, h = coords
                    else:
                        xs, ys = coords[0::2], coords[1::2]
                        if not xs: continue
                        x_min, x_max = min(xs), max(xs)
                        y_min, y_max = min(ys), max(ys)
                        cx = (x_min + x_max) / 2
                        cy = (y_min + y_max) / 2
                        w = x_max - x_min
                        h = y_max - y_min
                    det_lines.append(f"{crack_class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
                
                with open(dst_lbl_file, 'w') as f:
                    f.writelines(det_lines)
            
            merged_count += 1
    
    print(f"✅ Merged {merged_count} crack images into training set")
else:
    print("Skipping crack merge (set MERGE_CRACK_DATA=True to enable)")

## 2c. Create Proper Train/Val Split

The Roboflow dataset has all images in train. We need a validation set.

In [ ]:
# Create 80/20 train/val split
train_img_dir = ROBOFLOW_DIR / 'train' / 'images'
train_lbl_dir = ROBOFLOW_DIR / 'train' / 'labels'
valid_img_dir = ROBOFLOW_DIR / 'valid' / 'images'
valid_lbl_dir = ROBOFLOW_DIR / 'valid' / 'labels'

valid_img_dir.mkdir(parents=True, exist_ok=True)
valid_lbl_dir.mkdir(parents=True, exist_ok=True)

existing_valid = len(list(valid_img_dir.glob('*.[jJ][pP][gG]'))) + len(list(valid_img_dir.glob('*.[pP][nN][gG]')))
train_images = sorted([f for f in train_img_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])

val_ratio = 0.2
needed = int(len(train_images) * val_ratio / (1 - val_ratio)) - existing_valid

if needed > 0:
    random.seed(42)
    random.shuffle(train_images)
    for img_file in train_images[:needed]:
        shutil.move(str(img_file), str(valid_img_dir / img_file.name))
        lbl = train_lbl_dir / (img_file.stem + '.txt')
        if lbl.exists():
            shutil.move(str(lbl), str(valid_lbl_dir / lbl.name))

# Update data.yaml with correct paths
config['train'] = str(ROBOFLOW_DIR / 'train' / 'images')
config['val'] = str(ROBOFLOW_DIR / 'valid' / 'images')
if 'test' not in config or not (ROBOFLOW_DIR / 'test' / 'images').exists():
    config['test'] = config['val']  # use val as test

with open(data_yaml, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Print stats
n_train = len(list((ROBOFLOW_DIR / 'train' / 'images').iterdir()))
n_valid = len(list(valid_img_dir.iterdir()))
print(f"\n📊 Dataset ready:")
print(f"   Train: {n_train} images")
print(f"   Valid: {n_valid} images")
print(f"   Classes: {config['names']}")
print(f"\n   data.yaml: {data_yaml}")

## 3. Train YOLOv8n (Detection)

In [ ]:
# Load YOLOv8 NANO for detection (not seg)
model = YOLO('yolov8n.pt')

device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if torch.cuda.is_available() else 4
epochs = 80  # More epochs since small dataset

print(f"Device: {device} | Epochs: {epochs} | Batch: {batch_size}")
print(f"Data: {data_yaml}")

results = model.train(
    data=str(data_yaml),
    epochs=epochs,
    imgsz=640,
    batch=batch_size,
    device=device,
    patience=20,
    save=True,
    save_period=10,
    # Aggressive augmentation (small dataset needs this)
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    flipud=0.2,
    degrees=15.0,
    translate=0.15,
    scale=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.3,
    name="smartlease_multiclass",
    project="runs/train"
)

best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"\n✅ Training complete!")
print(f"Best weights: {best_weights}")

## 4. Evaluate Performance

In [ ]:
# Plot training curves
results_img = Path(results.save_dir) / 'results.png'
if results_img.exists():
    img = cv2.cvtColor(cv2.imread(str(results_img)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves', fontsize=14)
    plt.show()

# Confusion matrix
cm_img = Path(results.save_dir) / 'confusion_matrix_normalized.png'
if cm_img.exists():
    img = cv2.cvtColor(cv2.imread(str(cm_img)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix', fontsize=14)
    plt.show()

# Validation metrics
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=str(data_yaml))
print(f"\n📊 Validation Results:")
print(f"   mAP@50:    {metrics.box.map50:.4f}")
print(f"   mAP@50-95: {metrics.box.map:.4f}")
print(f"   Precision:  {metrics.box.mp:.4f}")
print(f"   Recall:     {metrics.box.mr:.4f}")

## 5. Visual Test Inference

In [ ]:
# Show validation predictions
val_preds = list(Path(results.save_dir).glob('val_batch*_pred.jpg'))
if val_preds:
    fig, axes = plt.subplots(1, min(2, len(val_preds)), figsize=(20, 10))
    if len(val_preds) == 1:
        axes = [axes]
    for ax, pred in zip(axes, val_preds[:2]):
        img = cv2.cvtColor(cv2.imread(str(pred)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(pred.name)
    plt.tight_layout()
    plt.show()

# Run inference on a few test images
test_dir = ROBOFLOW_DIR / 'valid' / 'images'
test_images = list(test_dir.glob('*.jpg'))[:6]

if test_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, img_path in zip(axes.flat, test_images):
        result = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
        annotated = result.plot()
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        n_det = len(result.boxes)
        ax.set_title(f"{n_det} detection(s)", fontsize=10)
    plt.suptitle('Inference on Validation Images', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Export & Download

In [ ]:
# Export to ONNX for NPU deployment
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=False
)
print(f"\n✅ ONNX exported: {onnx_path}")

# Copy best weights to a convenient location
shutil.copy2(str(best_weights), '/content/best_multiclass.pt')
print(f"\n📥 Download your trained model:")
print(f"   PyTorch: /content/best_multiclass.pt")
print(f"   ONNX:    {onnx_path}")
print(f"\n   Copy to your project: models/vision_best.pt")

In [ ]:
# Download files (Colab only)
try:
    from google.colab import files
    files.download('/content/best_multiclass.pt')
    print("Downloading best_multiclass.pt...")
except:
    print("Not in Colab — copy the file manually from the path above")